In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [12]:
class LogisticRegression:
    def __init__(self):
        self.optimizer = None
        self.hat_m_b = None
        self.hat_m_w = None
        self.r_v_b = None
        self.r_v_w = None
        self.hat_v_w = None
        self.hat_v_b = None
        self.x = None
        self.y = None
        self._y = None
        self.w = None
        self.b = None
        self.w_error = None
        self.b_error = None
        self.m_w = None
        self.m_b = None
        self.v_w = None
        self.v_b = None
        self.epoch_count = 1
        self.epoch_limit = None
    def predict(self,x):
        out = 1/(1 + np.exp(-(np.dot(x , self.w) + self.b)))
        return out
    def fit(self,x,y,epochs = 100,optimizer = 'gd'):
        self.x = x
        self.y = y
        self.w = np.zeros(x.shape[1])
        self.b = 0
        self._y = self.predict(self.x)
        self.epoch_limit = epochs
        self.optimizer = optimizer
        #initialized the momentum to avoid type error in first iteration
        self.v_w = np.zeros(self.x.shape[1])
        self.m_w = np.zeros(self.x.shape[1])
        self.m_b = 0
        self.v_b = 0
        #####
        self.b_error = -(np.mean(self.y - self._y))
        self.w_error = -(np.dot(self.x.transpose(),self.y - self._y))/self.x.shape[0]
        if self.optimizer == "adam":
            update_step = self.adam
        elif self.optimizer == "RMSprop":
            update_step = self.RMSprop
        elif self.optimizer == "momentum":
            update_step = self.momentum
        elif self.optimizer == "gd":
            update_step = self.gd
        else:
            raise ValueError(f"Unknown optimizer: {self.optimizer}")
        while self.epoch_count <= self.epoch_limit:
            self._y = self.predict(self.x)
            self.b_error = -(np.mean(self.y - self._y))
            self.w_error = -(np.dot(self.x.transpose(),self.y - self._y))/self.x.shape[0]
            update_step()
            print(f'Epoch : {self.epoch_count} Loss : ')
            self.epoch_count += 1
    def adam(self, beta1 = 0.9 , beta2 = 0.999 ,epsilon = 10**-8,gamma = 0.0003):
        #momentum variables
        self.m_w = beta1*self.m_w + (1 - beta1)*self.w_error
        self.m_b = beta1*self.m_b + (1 - beta1)*self.b_error
        #RMSpropFactors
        self.v_w = beta2*self.v_w + (1-beta2)*(self.w_error**2)
        self.v_b = beta2*self.v_b + (1-beta2)*(self.b_error**2)
        #bias correcting the variance values
        self.hat_v_w = self.v_w / (1 - beta2**self.epoch_count)
        self.hat_v_b = self.v_b / (1 - beta2**self.epoch_count)
        #RMSpropFactvectors for updating the final values
        self.r_v_w = 1 / (np.sqrt(self.hat_v_w + epsilon))
        self.r_v_b = 1 / (np.sqrt(self.hat_v_b + epsilon))
        #bias correcting the momentum values
        self.hat_m_w = self.m_w / (1 - beta1**self.epoch_count)
        self.hat_m_b = self.m_b / (1 - beta1**self.epoch_count)
        #Updating the final value
        self.b = self.b - self.hat_m_b*self.r_v_b*gamma
        self.w = self.w - np.multiply(self.hat_m_w,self.r_v_w)*gamma
    def RMSprop(self,beta2 = 0.999,epsilon = 10**-8, gamma = 0.001):
        self.v_w = beta2*self.v_w + (1-beta2)*(self.w_error**2)
        self.v_b = beta2*self.v_b + (1-beta2)*(self.b_error**2)
        self.r_v_w = 1 / (np.sqrt(self.v_w + epsilon))
        self.r_v_b = 1 / (np.sqrt(self.v_b + epsilon))
        self.b = self.b - self.b_error*self.r_v_b*gamma
        self.w = self.w - np.multiply(self.w_error,self.r_v_w)*gamma
    def momentum(self,beta1 = 0.9 , gamma = 0.001):
        self.m_w = beta1*self.m_w + (1 - beta1)*self.w_error
        self.m_b = beta1*self.m_b + (1 - beta1)*self.b_error
        self.hat_m_w = self.m_w / (1 - beta1**self.epoch_count)
        self.hat_m_b = self.m_b / (1 - beta1**self.epoch_count)
        self.b = self.b - self.hat_m_b*gamma
        self.w = self.w - self.hat_m_w*gamma
    def gd(self,gamma = 0.005):
        self.w = self.w - gamma*self.w_error
        self.b = self.b - self.b_error*gamma

In [13]:
df = pd.read_csv('Iris.csv')
training_df = df[df.Species == 'Iris-setosa']
x_variable = training_df.loc[:,['SepalLengthCm','SepalWidthCm']]
y_variable = training_df.loc[:,['PetalLengthCm']]
x_train,x_test,y_train,y_test = train_test_split(x_variable, y_variable, test_size = 0.10)

In [14]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform it
x_train_scaled = scaler.fit_transform(x_train.values)

# --- IMPORTANT ---
# Use that *same* scaler to transform your test data
x_test_scaled = scaler.transform(x_test.values)

# Now, train your model on the SCALED data
model = LogisticRegression()
model.fit(x_train_scaled, y_train.values.ravel(), epochs = 1500,optimizer = "gd")
y_pred = model.predict(x_test_scaled)

# And predict on the SCALED test data
predictions = model.predict(x_test_scaled)
print(predictions)
#
# # Now plot your results
# plt.scatter(y_test, predictions)
# print(model.class_class())



Epoch : 1 Loss : 
Epoch : 2 Loss : 
Epoch : 3 Loss : 
Epoch : 4 Loss : 
Epoch : 5 Loss : 
Epoch : 6 Loss : 
Epoch : 7 Loss : 
Epoch : 8 Loss : 
Epoch : 9 Loss : 
Epoch : 10 Loss : 
Epoch : 11 Loss : 
Epoch : 12 Loss : 
Epoch : 13 Loss : 
Epoch : 14 Loss : 
Epoch : 15 Loss : 
Epoch : 16 Loss : 
Epoch : 17 Loss : 
Epoch : 18 Loss : 
Epoch : 19 Loss : 
Epoch : 20 Loss : 
Epoch : 21 Loss : 
Epoch : 22 Loss : 
Epoch : 23 Loss : 
Epoch : 24 Loss : 
Epoch : 25 Loss : 
Epoch : 26 Loss : 
Epoch : 27 Loss : 
Epoch : 28 Loss : 
Epoch : 29 Loss : 
Epoch : 30 Loss : 
Epoch : 31 Loss : 
Epoch : 32 Loss : 
Epoch : 33 Loss : 
Epoch : 34 Loss : 
Epoch : 35 Loss : 
Epoch : 36 Loss : 
Epoch : 37 Loss : 
Epoch : 38 Loss : 
Epoch : 39 Loss : 
Epoch : 40 Loss : 
Epoch : 41 Loss : 
Epoch : 42 Loss : 
Epoch : 43 Loss : 
Epoch : 44 Loss : 
Epoch : 45 Loss : 
Epoch : 46 Loss : 
Epoch : 47 Loss : 
Epoch : 48 Loss : 
Epoch : 49 Loss : 
Epoch : 50 Loss : 
Epoch : 51 Loss : 
Epoch : 52 Loss : 
Epoch : 53 Loss : 
Ep

In [ ]:
class MulticlassClassification:
    def __init__(self):
        self.models = []

    def fit(self, X, y):
        for y_i in np.unique(y):
            x_true = X[y == y_i]
            x_false = X[y != y_i]
            x_true_false = np.vstack((x_true, x_false))
            y_true = np.ones(x_true.shape[0])
            y_false = np.zeros(x_false.shape[0])
            y_true_false = np.hstack((y_true, y_false))
            model = LogisticRegression()
            model.fit(x_true_false, y_true_false)
            self.models.append([y_i, model])
    def predict(self, X):
        y_pred = [[label, model.predict(X)] for label, model in self.models]

        output = []

        for i in range(X.shape[0]):
            max_label = None
            max_prob = -10**5
            for j in range(len(y_pred)):
                prob = y_pred[j][1][i]
                if prob > max_prob:
                    max_label = y_pred[j][0]
                    max_prob = prob
            output.append(max_label)

        return output